<a href="https://colab.research.google.com/github/hannaginther/ENGG680_2025_Fall/blob/main/Project/Group_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ENGG680 - Introduction to Digital Engineering**
## *Group Project: Implementing ML model for Hospital Length of Stay Prediction*

## Preliminary: Certificate of Work
### (5 Marks)

*We, the undersigned, certify that this is our own work, which has been done expressly for this course, either without the assistance of any other party or where appropriate we have acknowledged the work of others. Further, we have read and understood the section in the university calendar on plagiarism/cheating/other academic misconduct and we are aware of the implications thereof. We request that the total mark for this assignment be distributed as follows among group members:*

|          | First Name | Last Name | Signature (Full Name, Date) | Hours | Contribution % |
|----------|------------|-----------|-----------------------------|-------|----------------|
| Member 1: | Allison | Haynes | Signature, Date | hrs | 33.33% |
| Member 2: | Diya | Chakraborty | Signature, Date | hrs | 33.33% |
| Member 3: | Hanna | Ginther | Signature, Date | hrs | 33.33% |


##**Step #1: Data Cleaning & Preprocessing:**

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

In [2]:
sparcs_2024 = pd.read_csv('/content/drive/MyDrive/Sparcs_Datafiles/Hospital_Inpatient_Discharges_(SPARCS_De-Identified)__2024_20251016.csv')

/tmp/ipython-input-2869369836.py:1: DtypeWarning: Columns (29) have mixed types. Specify dtype option on import or set low_memory=False.
  sparcs_2024 = pd.read_csv('/content/drive/MyDrive/Sparcs_Datafiles/Hospital_Inpatient_Discharges_(SPARCS_De-Identified)__2024_20251016.csv')


In [4]:
sparcs_2024.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2196737 entries, 0 to 2196736
Data columns (total 33 columns):
 #   Column                               Dtype  
---  ------                               -----  
 0   Health Service Area                  object 
 1   Hospital County                      object 
 2   Operating Certificate Number         float64
 3   Permanent Facility Id                float64
 4   Facility Name                        object 
 5   Age Group                            object 
 6   Zip Code                             object 
 7   Gender                               object 
 8   Race                                 object 
 9   Ethnicity                            object 
 10  Length of Stay                       object 
 11  Type of Admission                    object 
 12  Patient Disposition                  object 
 13  Discharge Year                       int64  
 14  CCSR Diagnosis Code                  object 
 15  CCSR Diagnosis Description      

In [6]:
sparcs_2024.head()

,Health Service Area,Hospital County,Operating Certificate Number,Permanent Facility Id,Facility Name,Age Group,Zip Code,Gender,Race,Ethnicity,...,APR Severity of Illness Description,APR Risk of Mortality,APR Medical Surgical Description,Payment Typology 1,Payment Typology 2,Payment Typology 3,Birth Weight,Emergency Department Indicator,Total Charges,Total Costs
0,Hudson Valley,Westchester,5957001.0,1139.0,WESTCHESTER MEDICAL CENTER,0-17,OOS,F,White,Not Span/Hispanic,...,Moderate,Minor,Medical,Private Health Insurance,NaN,NaN,NaN,Y,46814.00,6772.07
1,New York City,Queens,7003001.0,1628.0,FLUSHING HOSPITAL MEDICAL CENTER,0-17,113,M,White,Spanish/Hispanic,...,Moderate,Moderate,Medical,Medicaid,NaN,NaN,NaN,Y,13490.00,15464.30
2,New York City,New York,7002054.0,1458.0,NEW YORK-PRESBYTERIAN HOSPITAL - NEW YORK WEIL...,70 or Older,100,M,White,Not Span/Hispanic,...,Moderate,Moderate,Medical,Medicare,Private Health Insurance,NaN,NaN,Y,49503.16,9324.77
3,New York City,New York,7002054.0,1464.0,NEW YORK-PRESBYTERIAN HOSPITAL - COLUMBIA PRES...,0-17,100,F,Other Race,Not Span/Hispanic,...,Minor,Minor,Medical,Private Health Insurance,NaN,NaN,2700,Y,27827.66,7304.27
4,New York City,New York,7002032.0,1466.0,MOUNT SINAI WEST,18-29,100,F,Other Race,Spanish/Hispanic,...,Moderate,Minor,Medical,Medicare,NaN,NaN,NaN,Y,32798.29,7948.10


,Health Service Area,Hospital County,Operating Certificate Number,Permanent Facility Id,Facility Name,Age Group,Zip Code,Gender,Race,Ethnicity,...,APR Severity of Illness Description,APR Risk of Mortality,APR Medical Surgical Description,Payment Typology 1,Payment Typology 2,Payment Typology 3,Birth Weight,Emergency Department Indicator,Total Charges,Total Costs
2196732,New York City,New York,7002024.0,1456.0,MOUNT SINAI HOSPITAL,70 or Older,111,F,Multi-racial,Not Span/Hispanic,...,Moderate,Moderate,Surgical,Medicare,Medicare,Medicaid,NaN,N,59727.81,17084.60
2196733,Central NY,Jefferson,2238700.0,379.0,CARTHAGE AREA HOSPITAL INC,18-29,136,F,White,Not Span/Hispanic,...,Minor,Minor,Surgical,Federal/State/Local/VA,NaN,NaN,NaN,N,25474.57,43365.68
2196734,New York City,Kings,7001016.0,1301.0,KINGS COUNTY HOSPITAL CENTER,30-49,112,F,Black/African American,Unknown,...,Moderate,Minor,Surgical,Medicaid,NaN,NaN,NaN,N,51617.21,28211.54
2196735,Long Island,Nassau,7002053.0,511.0,NYU LANGONE HOSPITAL-LONG ISLAND,30-49,117,F,White,Not Span/Hispanic,...,Major,Moderate,Medical,Blue Cross/Blue Shield,NaN,NaN,NaN,N,60404.55,19568.72
2196736,New York City,Kings,7001035.0,1318.0,WYCKOFF HEIGHTS MEDICAL CENTER,30-49,113,F,Other Race,Spanish/Hispanic,...,Minor,Minor,Medical,Private Health Insurance,NaN,NaN,NaN,Y,6179.07,2238.09


In [16]:
#view columns 15-25 of the sparcs_2024 dataframe, with rows 10-30

sparcs_2024.iloc[10:30, 14:26]

,CCSR Diagnosis Code,CCSR Diagnosis Description,CCSR Procedure Code,CCSR Procedure Description,APR DRG Code,APR DRG Description,APR MDC Code,APR MDC Description,APR Severity of Illness Code,APR Severity of Illness Description,APR Risk of Mortality,APR Medical Surgical Description
10,RSP009,ASTHMA,ADM021,"ADMINISTRATION OF THERAPEUTIC SUBSTANCES, NEC",141,ASTHMA,4,DISEASES AND DISORDERS OF THE RESPIRATORY SYSTEM,1,Minor,Minor,Medical
11,END009,OBESITY,GIS010,GASTRECTOMY,403,PROCEDURES FOR OBESITY,10,ENDOCRINE NUTRITIONAL AND METABOLIC DISEASES A...,1,Minor,Minor,Surgical
12,END009,OBESITY,GIS010,GASTRECTOMY,403,PROCEDURES FOR OBESITY,10,ENDOCRINE NUTRITIONAL AND METABOLIC DISEASES A...,1,Minor,Minor,Surgical
13,END009,OBESITY,GIS010,GASTRECTOMY,403,PROCEDURES FOR OBESITY,10,ENDOCRINE NUTRITIONAL AND METABOLIC DISEASES A...,2,Moderate,Minor,Surgical
14,END009,OBESITY,ADM001,TRANSFUSION OF BLOOD AND BLOOD PRODUCTS,421,"MALNUTRITION, FAILURE TO THRIVE AND OTHER NUTR...",10,ENDOCRINE NUTRITIONAL AND METABOLIC DISEASES A...,3,Major,Moderate,Medical
15,END009,OBESITY,GIS019,GASTRO-JEJUNAL BYPASS (INCLUDING BARIATRIC),403,PROCEDURES FOR OBESITY,10,ENDOCRINE NUTRITIONAL AND METABOLIC DISEASES A...,2,Moderate,Minor,Surgical
16,END009,OBESITY,GIS019,GASTRO-JEJUNAL BYPASS (INCLUDING BARIATRIC),403,PROCEDURES FOR OBESITY,10,ENDOCRINE NUTRITIONAL AND METABOLIC DISEASES A...,1,Minor,Minor,Surgical
17,END009,OBESITY,GIS019,GASTRO-JEJUNAL BYPASS (INCLUDING BARIATRIC),403,PROCEDURES FOR OBESITY,10,ENDOCRINE NUTRITIONAL AND METABOLIC DISEASES A...,1,Minor,Minor,Surgical
18,SYM001,SYNCOPE,NaN,NaN,204,SYNCOPE AND COLLAPSE,5,DISEASES AND DISORDERS OF THE CIRCULATORY SYSTEM,2,Moderate,Minor,Medical
19,SYM001,SYNCOPE,NaN,NaN,204,SYNCOPE AND COLLAPSE,5,DISEASES AND DISORDERS OF THE CIRCULATORY SYSTEM,2,Moderate,Minor,Medical


In [19]:
num_missing = sparcs_2024['Birth Weight'].isna().sum()
print(f"Number of missing Birth Weight values: {num_missing}")

percent_missing = sparcs_2024['Birth Weight'].isna().mean() * 100
print(f"Percentage of missing Birth Weight values: {percent_missing:.2f}%")

Number of missing Birth Weight values: 1988915
Percentage of missing Birth Weight values: 90.54%


In [21]:
# Defining a function to preprocess sparcs_2024 dataframe:
import pandas as pd
import numpy as np
import json

def preprocess_sparcs (df):
  """
  Comprehensive preprocessing for SPARCS dataset.

  Steps included:
  1. Handle missing values (NaN)
    - Categorical columns -> Replace with 'Unknown'
    - Continuous numeric columns -> Replace with median
    - Code/ID numeric columns -> Leave NaN (numeric) for now
  2. Build clinical code/description mappings + drop description columns
    - NaN replaced with placeholder in mapping dict
  3. Build Permanent Facility Id/Facility Name mapping + drop Facility Name column
    - NaN replaced with placeholder in mapping dict
  4. Drop unnecessary Operating Certificate Number column (not used)
  5. Convert dtypes for ML
  """

  df = df.copy()  # Make a copy to avoid modifying original DataFrame:

  #---------------------------
  # Handle missing values first
  #---------------------------
  # Replace NaN in categorical columns with 'Unknown'
  categorical_cols = df.select_dtypes(include='object').columns.tolist()
  for col in categorical_cols:
    df[col] = df[col].fillna('Unknown')

  # Replace NaN in continuous numeric columns
  continuous_numeric_cols = ['Total Charges', 'Total Costs']  #Continuous values
  for col in continuous_numeric_cols:
    if col in df.columns:
      df[col] = df[col].fillna(df[col].median())

  #---------------------------
  # Clinical code/description columns mapping
  #---------------------------
  possible_code_cols = [col for col in df.columns if 'code' in col.lower()]
  exclude_terms = ['zip', 'postal', 'zip code', 'permanent facility id']
  code_cols = [col for col in possible_code_cols if not any (term in col.lower() for term in exclude_terms)]

  desc_cols = [col for col in df.columns if 'description' in col.lower()]

  code_mappings = {}
  mapped_desc_cols = []

  for code_col in code_cols:
    # Find corresponding description column
    desc_col = next((d for d in desc_cols if code_col.lower().replace('code', '') in d.lower()), None)
    if desc_col:
      # Temporarily replace NaN with placeholder for dictionary only
      temp_code_col = df[code_col].fillna(-1)
      temp_desc_col = df[desc_col].fillna('Unknown')

      # Create dictionary for mapping
      mapping_dict = dict(zip(temp_code_col, temp_desc_col))
      # Replace placeholder key with 'Unknown'
      mapping_dict = {k if k != -1 else 'Unknown': v for k, v in mapping_dict.items()}

      # Add mapping to dictionary
      code_mappings[code_col] = mapping_dict

      # Append mapped description column to list
      mapped_desc_cols.append(desc_col)

  #Drop mapped description columns
  df.drop(columns=mapped_desc_cols, inplace=True)

  #---------------------------
  # Permanent Facility ID/Facility Name columns mapping
  #---------------------------
  facility_mapping = {}
  if 'Permanent Facility Id' in df.columns and 'Facility Name' in df.columns:
    # Temporarily replace NaN with placeholder for dictionary only
    temp_facility_id = df['Permanent Facility Id'].fillna(-1)
    temp_facility_name = df['Facility Name'].fillna('Unknown')

    # Create dictionary for mapping
    facility_mapping = dict(zip(temp_facility_id, temp_facility_name))
    # Replace placeholder key with 'Unknown'
    facility_mapping = {k if k != -1 else 'Unknown': v for k, v in facility_mapping.items()}
    # Drop Facility Name column from DataFram
    df.drop(columns=['Facility Name'], inplace=True)

  #---------------------------
  # Drop Operating Certificate Number column
  #---------------------------
  df.drop(columns=['Operating Certificate Number'], inplace=True)

  #---------------------------
  # Convert data types for ML
  #---------------------------

  # Categorical columns:
  for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype('category')

  # Continuous numeric columns:
  for col in continuous_numeric_cols:
    if col in df.columns:
      df[col] = df[col].astype('float32')

  # Identifier / code numeric columns -> category:
  id_code_cols = [
      'Discharge Year', 'APR DRG Code', 'APR MDC Code', 'APR Severity of Illness Code', 'Permanent Facility Id'
  ]

  for col in id_code_cols:
    if col in df.columns:
      df[col] = df[col].astype('category')

  return df, code_mappings, facility_mapping

  #---------------------------
  # Usage
  #---------------------------

  sparcs_2024_cleaned, sparcs_2024_code_mappings, sparcs_2024_facility_mapping = preprocess_sparcs(sparcs_2024)

  print ("Columns after preprocessing: ", sparcs_2024_cleaned.columns.tolist())
  print ("Sample clinical code mappings: ", json.dumps(dict(list(sparcs_2024_code_mappings.items())[:5], indent=2)))
  print ("Sample permanent facility mappings: ", json.dumps(dict(list(sparcs_2024_facility_mapping.items())[:5], indent=2)))
